1. Multithreading
Uses multiple threads within one process.
Useful for:
- API requests
- Database queries
- File operations
- Network calls

In [1]:
import threading
import time

def task():
    time.sleep(2)
    print("Task completed")

threads = []

for _ in range(3):
    t = threading.Thread(target=task)
    threads.append(t)
    t.start()

for t in threads:
    t.join()

Task completedTask completed
Task completed



2. Multiprocessing
Uses multiple independent processes.
Useful for:
- CPU-intensive calculations
- Image processing
- Data processing
- Parallel computation

In [ ]:
import multiprocessing

def calculate():
    result = sum(i * i for i in range(10_000_000))
    print(result)

if __name__ == "__main__":
    processes = []

    for _ in range(2):
        p = multiprocessing.Process(target=calculate)
        processes.append(p)
        p.start()

    for p in processes:
        p.join()

3. Asyncio
Uses asynchronous programming with an event loop. It can manage many I/O operations without creating a separate thread for every task.


FastAPI applications



Concurrent HTTP requests



WebSockets



Database operations with async drivers

In [4]:
import asyncio

async def task(name):
    print(name, "started")
    await asyncio.sleep(2)
    print(name, "finished")

async def main():
    await asyncio.gather(
        task("Task 1"),
        task("Task 2"),
        task("Task 3")
    )
await main()



Task 1 started
Task 2 started
Task 3 started
Task 1 finished
Task 2 finished
Task 3 finished


## Mutex
A Lock ensures that only one thread at a time can enter a protected section of code.

In [5]:
import threading

lock = threading.Lock()

def task(name):
    with lock:
        print(name, "entered")
        print(name, "working")
        print(name, "exited")

t1 = threading.Thread(target=task, args=("Thread 1",))
t2 = threading.Thread(target=task, args=("Thread 2",))

t1.start()
t2.start()

t1.join()
t2.join()

Thread 1 entered
Thread 1 working
Thread 1 exited
Thread 2 entered
Thread 2 working
Thread 2 exited


Consider two functions. Function A calls Function B, and both need to protect shared data.

In [ ]:
import threading

lock = threading.Lock()

def function_b():
    with lock:
        print("Inside function B")

def function_a():
    with lock:
        print("Inside function A")
        function_b()

t1 = threading.Thread(target=function_a)
t1.start()
t1.join()

Inside function A


What happens?
1. T1 enters function_a() and acquires the lock.
2. T1 calls function_b().
3. function_b() tries to acquire the same lock.
4. But T1 already holds it.
5. T1 waits for the lock to be released.
6. The lock cannot be released because T1 is stuck inside function_b().

The drawback: A normal Lock does not allow the same thread to acquire it again before releasing it.

### RLock- Reetrant Lock

It allows the same thread that already owns a lock to acquire it repeatedly without blocking itself.

Other threads still have to wait until the owner releases all its acquisitions.

In [1]:
import threading

lock = threading.RLock()

def function_b():
    with lock:
        print("Inside function B")

def function_a():
    with lock:
        print("Inside function A")
        function_b()

t1 = threading.Thread(target=function_a)

t1.start()
t1.join()

print("Main thread finished")

Inside function A
Inside function B
Main thread finished


It tracks the owner thread and a recursion count.


Operation	Recursion count


T1 acquires in A   ,count  -   	1


T1 acquires in B	, count -      2


B releases,count -	            1


A releases	  , count -           0

Only when the count reaches zero can another thread acquire the lock.